# Toy Stable Diffusion with VAE, U-Net, Text Prompts, and Classifier-Free Guidance

This notebook is a **teaching-oriented but more realistic** latent diffusion implementation.

It adds the requested components:
- **VAE-style encoder/decoder**
- **U-Net architecture**
- **real text prompts**
- **classifier-free guidance**
- **training on image-text pairs**

To keep the notebook small and trainable in Colab or Jupyter, it uses **CIFAR-10** images and converts class labels into short natural-language prompts such as:
- `"a photo of an airplane"`
- `"a photo of a dog"`
- `"a photo of a truck"`

This is still much smaller than production Stable Diffusion, but the pipeline is substantially closer.


## Step 1: Imports


In [ ]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Step 2: Hyperparameters

These values are intentionally small so the notebook can run on modest hardware.


In [ ]:
@dataclass
class Config:
    image_size: int = 32
    batch_size: int = 128
    latent_channels: int = 4
    latent_size: int = 8
    text_embed_dim: int = 128
    time_embed_dim: int = 128
    unet_base_channels: int = 64
    num_timesteps: int = 200
    lr_vae: float = 1e-3
    lr_diffusion: float = 1e-3
    vae_epochs: int = 8
    diffusion_epochs: int = 12
    kl_weight: float = 1e-4
    cfg_dropout_prob: float = 0.1
    sample_cfg_scale: float = 3.0
    max_token_length: int = 8

cfg = Config()
cfg


## Step 3: Image-Text Pair Dataset

Stable Diffusion trains on image-text pairs.

For a lightweight classroom example, we convert CIFAR-10 labels into short prompts.


In [ ]:
cifar_class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def label_to_prompt(label):
    name = cifar_class_names[label]
    article = "an" if name[0] in "aeiou" else "a"
    return f"{article} photo of {article} {name}"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

base_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

class PromptedCIFAR10(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        prompt = label_to_prompt(label)
        return image, prompt, label

dataset = PromptedCIFAR10(base_dataset)
print("Number of training pairs:", len(dataset))

for i in range(5):
    _, prompt, label = dataset[i]
    print(label, "->", prompt)


## Step 4: Tiny Tokenizer

A real Stable Diffusion system uses a pretrained text encoder.

Here we use a tiny word-level tokenizer and embedding model so the full pipeline remains visible.


In [ ]:
special_tokens = ["<pad>", "<null>", "<unk>"]

all_prompts = [label_to_prompt(i) for i in range(10)]
vocab_words = set()
for p in all_prompts:
    for w in p.lower().split():
        vocab_words.add(w)

itos = special_tokens + sorted(vocab_words)
stoi = {w: i for i, w in enumerate(itos)}

PAD_ID = stoi["<pad>"]
NULL_ID = stoi["<null>"]
UNK_ID = stoi["<unk>"]

def encode_prompt(prompt, max_len=cfg.max_token_length):
    words = prompt.lower().split()
    token_ids = [stoi.get(w, UNK_ID) for w in words][:max_len]
    token_ids += [PAD_ID] * (max_len - len(token_ids))
    return token_ids

def encode_null_prompt(max_len=cfg.max_token_length):
    return [NULL_ID] + [PAD_ID] * (max_len - 1)

print("Vocabulary:", itos)
print("Example tokens:", encode_prompt("a photo of a dog"))
print("Null tokens:", encode_null_prompt())


## Step 5: DataLoader with Tokenized Prompts


In [ ]:
def collate_fn(batch):
    images, prompts, labels = zip(*batch)
    images = torch.stack(images)
    token_ids = torch.tensor([encode_prompt(p) for p in prompts], dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return images, token_ids, labels, list(prompts)

loader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device.type == "cuda" else False,
    collate_fn=collate_fn
)

images, token_ids, labels, prompts = next(iter(loader))
print(images.shape, token_ids.shape, labels.shape)
print(prompts[:4])


## Step 6: Visualization Helpers


In [ ]:
def to_display(x):
    x = x.detach().cpu()
    x = (x + 1) / 2
    return x.clamp(0, 1)

def show_images(images, titles=None, n=8):
    images = images[:n]
    images = to_display(images)
    fig, axes = plt.subplots(1, n, figsize=(2.2*n, 2.5))
    if n == 1:
        axes = [axes]
    for i in range(n):
        img = images[i]
        img = img.permute(1, 2, 0)
        axes[i].imshow(img)
        axes[i].axis("off")
        if titles is not None:
            axes[i].set_title(str(titles[i]), fontsize=9)
    plt.tight_layout()
    plt.show()


## Step 7: Text Encoder

This is a tiny text encoder:
- token embedding
- positional embedding
- mean pooling over non-pad tokens


In [ ]:
class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.max_len = max_len

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.SiLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, token_ids):
        # token_ids: [B, L]
        B, L = token_ids.shape
        positions = torch.arange(L, device=token_ids.device).unsqueeze(0).expand(B, L)
        x = self.token_embed(token_ids) + self.pos_embed(positions)

        mask = (token_ids != PAD_ID).float().unsqueeze(-1)
        masked_x = x * mask
        denom = mask.sum(dim=1).clamp(min=1.0)
        pooled = masked_x.sum(dim=1) / denom
        return self.mlp(pooled)


## Step 8: VAE-Style Encoder/Decoder

This VAE does:
- encoder -> `mu`, `logvar`
- reparameterization
- decoder reconstruction

Loss:
\[
L_{VAE} = L_{recon} + \lambda_{KL} L_{KL}
\]


In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),   # 32 -> 16
            nn.SiLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 16 -> 8
            nn.SiLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.SiLU()
        )
        self.to_mu = nn.Conv2d(64, latent_channels, 3, padding=1)
        self.to_logvar = nn.Conv2d(64, latent_channels, 3, padding=1)

    def forward(self, x):
        h = self.net(x)
        mu = self.to_mu(h)
        logvar = self.to_logvar(h)
        return mu, logvar

class VAEDecoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(latent_channels, 64, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="nearest"),   # 8 -> 16
            nn.Conv2d(64, 32, 3, padding=1),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="nearest"),   # 16 -> 32
            nn.Conv2d(32, 3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

class TinyVAE(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.encoder = VAEEncoder(latent_channels)
        self.decoder = VAEDecoder(latent_channels)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar, z

def vae_loss_fn(recon, x, mu, logvar, kl_weight=cfg.kl_weight):
    recon_loss = F.mse_loss(recon, x)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss = recon_loss + kl_weight * kl
    return loss, recon_loss, kl


## Step 9: Train the VAE


In [ ]:
vae = TinyVAE(latent_channels=cfg.latent_channels).to(device)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=cfg.lr_vae)

for epoch in range(cfg.vae_epochs):
    vae.train()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0

    for images, token_ids, labels, prompts in loader:
        images = images.to(device)

        recon, mu, logvar, z = vae(images)
        loss, recon_loss, kl = vae_loss_fn(recon, images, mu, logvar)

        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_recon += recon_loss.item() * images.size(0)
        total_kl += kl.item() * images.size(0)

    n = len(dataset)
    print(
        f"VAE Epoch {epoch+1}/{cfg.vae_epochs} | "
        f"loss={total_loss/n:.6f} | recon={total_recon/n:.6f} | kl={total_kl/n:.6f}"
    )


## Step 10: Inspect VAE Reconstructions


In [ ]:
vae.eval()
images, token_ids, labels, prompts = next(iter(loader))
images = images.to(device)

with torch.no_grad():
    recon, mu, logvar, z = vae(images)

print("Original:")
show_images(images, titles=prompts, n=8)

print("Reconstruction:")
show_images(recon, titles=prompts, n=8)


## Step 11: Diffusion Schedule in Latent Space

We now diffuse the latent tensor rather than the image.

Forward process:
\[
z_t = \sqrt{\bar{\alpha}_t} z_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon
\]


In [ ]:
betas = torch.linspace(1e-4, 0.02, cfg.num_timesteps, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def extract(a, t, x_shape):
    out = a.gather(0, t)
    return out.view(-1, 1, 1, 1)

def q_sample(z0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(z0)
    sqrt_alpha_bar = extract(torch.sqrt(alpha_bars), t, z0.shape)
    sqrt_one_minus_alpha_bar = extract(torch.sqrt(1 - alpha_bars), t, z0.shape)
    zt = sqrt_alpha_bar * z0 + sqrt_one_minus_alpha_bar * noise
    return zt, noise


## Step 12: Time Embedding


In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.proj = nn.Sequential(
            nn.Linear(dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=t.device).float() / half
        )
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return self.proj(emb)


## Step 13: U-Net Blocks

This is a small U-Net-style architecture with:
- down path
- bottleneck
- up path
- skip connections
- conditioning from time + text embeddings


In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.cond_proj = nn.Linear(cond_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, cond):
        h = self.conv1(x)
        cond_term = self.cond_proj(cond).unsqueeze(-1).unsqueeze(-1)
        h = h + cond_term
        h = F.silu(h)
        h = self.conv2(h)
        h = F.silu(h)
        return h + self.skip(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.res = ResBlock(in_ch, out_ch, cond_dim)
        self.down = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)

    def forward(self, x, cond):
        h = self.res(x, cond)
        d = self.down(h)
        return h, d

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.res = ResBlock(out_ch + skip_ch, out_ch, cond_dim)

    def forward(self, x, skip, cond):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.res(x, cond)
        return x

class TinyUNet(nn.Module):
    def __init__(self, latent_channels=4, base_ch=64, cond_dim=128):
        super().__init__()
        self.in_conv = nn.Conv2d(latent_channels, base_ch, 3, padding=1)

        self.down1 = DownBlock(base_ch, base_ch, cond_dim)        # 8 -> 4
        self.down2 = DownBlock(base_ch, base_ch * 2, cond_dim)    # 4 -> 2

        self.mid1 = ResBlock(base_ch * 2, base_ch * 2, cond_dim)
        self.mid2 = ResBlock(base_ch * 2, base_ch * 2, cond_dim)

        self.up2 = UpBlock(base_ch * 2, base_ch * 2, base_ch, cond_dim)  # 2 -> 4
        self.up1 = UpBlock(base_ch, base_ch, base_ch, cond_dim)           # 4 -> 8

        self.out_conv = nn.Conv2d(base_ch, latent_channels, 3, padding=1)

    def forward(self, x, cond):
        x = self.in_conv(x)
        skip1, x = self.down1(x, cond)
        skip2, x = self.down2(x, cond)

        x = self.mid1(x, cond)
        x = self.mid2(x, cond)

        x = self.up2(x, skip2, cond)
        x = self.up1(x, skip1, cond)

        return self.out_conv(x)


## Step 14: Classifier-Free Guidance (CFG)

During training, we randomly replace the real prompt with a null prompt.

That teaches the network both:
- conditional denoising
- unconditional denoising

At sampling time we combine them:
\[
\hat{\epsilon} = \epsilon_{uncond} + s (\epsilon_{cond} - \epsilon_{uncond})
\]
where `s` is the guidance scale.


In [ ]:
def maybe_drop_condition(token_ids, drop_prob=cfg.cfg_dropout_prob):
    B = token_ids.size(0)
    null_tokens = torch.tensor(
        [encode_null_prompt() for _ in range(B)],
        device=token_ids.device,
        dtype=torch.long
    )
    drop_mask = (torch.rand(B, device=token_ids.device) < drop_prob).long().view(B, 1)
    return torch.where(drop_mask.bool(), null_tokens, token_ids)


## Step 15: Build the Diffusion Model


In [ ]:
text_encoder = TinyTextEncoder(
    vocab_size=len(itos),
    embed_dim=cfg.text_embed_dim,
    max_len=cfg.max_token_length
).to(device)

time_encoder = TimeEmbedding(cfg.time_embed_dim).to(device)

unet = TinyUNet(
    latent_channels=cfg.latent_channels,
    base_ch=cfg.unet_base_channels,
    cond_dim=cfg.text_embed_dim
).to(device)

diff_optimizer = torch.optim.Adam(
    list(text_encoder.parameters()) +
    list(time_encoder.parameters()) +
    list(unet.parameters()),
    lr=cfg.lr_diffusion
)

for p in vae.parameters():
    p.requires_grad = False

vae.eval()


## Step 16: Train the Latent Diffusion Model on Image-Text Pairs


In [ ]:
for epoch in range(cfg.diffusion_epochs):
    text_encoder.train()
    time_encoder.train()
    unet.train()

    total_loss = 0.0

    for images, token_ids, labels, prompts in loader:
        images = images.to(device)
        token_ids = token_ids.to(device)

        with torch.no_grad():
            mu, logvar = vae.encoder(images)
            z0 = mu  # use mean latent for stability

        t = torch.randint(0, cfg.num_timesteps, (images.size(0),), device=device)
        zt, noise = q_sample(z0, t)

        dropped_token_ids = maybe_drop_condition(token_ids, drop_prob=cfg.cfg_dropout_prob)

        text_cond = text_encoder(dropped_token_ids)
        time_cond = time_encoder(t)
        cond = text_cond + time_cond

        pred_noise = unet(zt, cond)
        loss = F.mse_loss(pred_noise, noise)

        diff_optimizer.zero_grad()
        loss.backward()
        diff_optimizer.step()

        total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / len(dataset)
    print(f"Diffusion Epoch {epoch+1}/{cfg.diffusion_epochs}, Loss = {avg_loss:.6f}")


## Step 17: Guided Reverse Diffusion Sampling

This cell uses:
- unconditional prediction
- conditional prediction
- classifier-free guidance


In [ ]:
@torch.no_grad()
def predict_noise_with_cfg(zt, t, prompt, guidance_scale=cfg.sample_cfg_scale):
    token_ids = torch.tensor([encode_prompt(prompt)], device=device, dtype=torch.long)
    null_ids = torch.tensor([encode_null_prompt()], device=device, dtype=torch.long)

    cond_text = text_encoder(token_ids)
    uncond_text = text_encoder(null_ids)
    time_cond = time_encoder(t)

    cond_pred = unet(zt, cond_text + time_cond)
    uncond_pred = unet(zt, uncond_text + time_cond)

    guided = uncond_pred + guidance_scale * (cond_pred - uncond_pred)
    return guided

@torch.no_grad()
def p_sample_cfg(zt, t, prompt, guidance_scale=cfg.sample_cfg_scale):
    beta_t = extract(betas, t, zt.shape)
    alpha_t = extract(alphas, t, zt.shape)
    alpha_bar_t = extract(alpha_bars, t, zt.shape)

    pred_noise = predict_noise_with_cfg(zt, t, prompt, guidance_scale)

    mean = (1 / torch.sqrt(alpha_t)) * (
        zt - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * pred_noise
    )

    noise = torch.randn_like(zt)
    nonzero_mask = (t != 0).float().view(-1, 1, 1, 1)
    sample = mean + nonzero_mask * torch.sqrt(beta_t) * noise
    return sample


## Step 18: Text-to-Image Sampling

You can type one of the prompt styles used during training, for example:
- `"a photo of a dog"`
- `"a photo of a truck"`
- `"a photo of an airplane"`


In [ ]:
@torch.no_grad()
def sample_from_prompt(prompt, guidance_scale=cfg.sample_cfg_scale, n=8):
    text_encoder.eval()
    time_encoder.eval()
    unet.eval()
    vae.eval()

    zt = torch.randn(n, cfg.latent_channels, cfg.latent_size, cfg.latent_size, device=device)

    prompt_list = [prompt] * n
    for step in reversed(range(cfg.num_timesteps)):
        t = torch.full((n,), step, device=device, dtype=torch.long)

        # batch CFG
        token_ids = torch.tensor([encode_prompt(prompt) for _ in range(n)], device=device, dtype=torch.long)
        null_ids = torch.tensor([encode_null_prompt() for _ in range(n)], device=device, dtype=torch.long)

        cond_text = text_encoder(token_ids)
        uncond_text = text_encoder(null_ids)
        time_cond = time_encoder(t)

        cond_pred = unet(zt, cond_text + time_cond)
        uncond_pred = unet(zt, uncond_text + time_cond)
        pred_noise = uncond_pred + guidance_scale * (cond_pred - uncond_pred)

        beta_t = extract(betas, t, zt.shape)
        alpha_t = extract(alphas, t, zt.shape)
        alpha_bar_t = extract(alpha_bars, t, zt.shape)

        mean = (1 / torch.sqrt(alpha_t)) * (
            zt - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * pred_noise
        )

        noise = torch.randn_like(zt)
        nonzero_mask = (t != 0).float().view(-1, 1, 1, 1)
        zt = mean + nonzero_mask * torch.sqrt(beta_t) * noise

    images = vae.decoder(zt)
    return images, prompt_list


## Step 19: Generate Samples from Real Text Prompts


In [ ]:
test_prompts = [
    "a photo of a dog",
    "a photo of a truck",
    "a photo of a cat",
    "a photo of an airplane"
]

for prompt in test_prompts:
    print("Prompt:", prompt)
    imgs, titles = sample_from_prompt(prompt, guidance_scale=3.0, n=6)
    show_images(imgs, titles=[prompt]*6, n=6)


## Step 20: Compare Guidance Scales

Classifier-free guidance changes how strongly the prompt steers generation.


In [ ]:
prompt = "a photo of a dog"

for scale in [0.0, 1.0, 3.0, 6.0]:
    print(f"Prompt: {prompt} | guidance_scale={scale}")
    imgs, _ = sample_from_prompt(prompt, guidance_scale=scale, n=6)
    show_images(imgs, titles=[f"cfg={scale}"]*6, n=6)


## Step 21: What This Notebook Now Includes

Compared with the earlier toy notebook, this version includes all requested ideas:

1. **U-Net architecture**  
   The denoiser is now a U-Net-style network with skip connections.

2. **Classifier-free guidance**  
   Training randomly drops prompts. Sampling combines conditional and unconditional predictions.

3. **Real text prompts**  
   Prompts are natural-language strings such as `"a photo of a dog"`.

4. **VAE-style encoder/decoder**  
   The latent representation now comes from a VAE with `mu`, `logvar`, KL loss, and reparameterization.

5. **Training on image-text pairs**  
   CIFAR-10 images are paired with text prompts, creating a lightweight image-text dataset.


## Step 22: Important Note

This notebook is **structurally closer to Stable Diffusion**, but still much smaller:

- text encoder is tiny rather than CLIP,
- VAE is tiny,
- U-Net is small,
- prompts are simple,
- dataset is CIFAR-10 rather than large web-scale image-text data.

That makes it suitable for teaching and experimentation.
